# ResNet-32 CIFAR-10 — Analog-Aware Training
**Laere Enterprises Analog Compute R&D**

This notebook reproduces IBM's Nature Communications methodology (Joshi et al., 2020):
- Gaussian noise injection on weights during forward pass (η = 3.8%)
- Weight clipping to [-2σ, +2σ]
- First/last layer protection
- Pretrained initialization + retrain with noise

**Target:** 93.7% accuracy on CIFAR-10 (ResNet-32)

Run this on Google Colab with GPU runtime (Runtime → Change runtime type → GPU)

In [ ]:
# ============================================
# Cell 1: Install dependencies
# ============================================
!pip install torch torchvision aihwkit -q
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'CUDA device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# ============================================
# Cell 2: ResNet-32 Architecture + Training
# ============================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import time

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes, 1, stride, bias=False),
                nn.BatchNorm2d(self.expansion*planes)
            )
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

class ResNet32(nn.Module):
    def __init__(self, num_classes=10, inject_noise=False, noise_eta=0.038, protect_first_last=True):
        super().__init__()
        self.inject_noise = inject_noise
        self.noise_eta = noise_eta
        self.protect_first_last = protect_first_last
        self.conv1 = nn.Conv2d(3, 16, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.layer1 = self._make_layer(16, 16, 5, 1)
        self.layer2 = self._make_layer(16, 32, 5, 2)
        self.layer3 = self._make_layer(32, 64, 5, 2)
        self.linear = nn.Linear(64, num_classes)
        self.apply(self._weights_init)

    def _make_layer(self, in_planes, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for s in strides:
            layers.append(BasicBlock(in_planes, planes, s))
            in_planes = planes * BasicBlock.expansion
        return nn.Sequential(*layers)

    def _weights_init(self, m):
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.constant_(m.weight, 1)
            nn.init.constant_(m.bias, 0)

    def _inject_noise(self, weight, eta):
        w_max = weight.abs().max()
        std = eta * w_max
        return weight + torch.randn_like(weight) * std

    def forward(self, x):
        # First conv (protected if configured)
        w = self.conv1.weight
        if self.inject_noise and not self.protect_first_last:
            w = self._inject_noise(w, self.noise_eta)
        out = F.conv2d(x, w, None, self.conv1.stride, self.conv1.padding)
        out = F.relu(self.bn1(out))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = F.avg_pool2d(out, out.size()[3])
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out

def train_epoch(model, loader, optimizer, criterion, device, inject_noise=False, noise_eta=0.038, clip_alpha=2.0):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch_idx, (data, target) in enumerate(loader):
        data, target = data.to(device), target.to(device)
        if hasattr(model, 'inject_noise'):
            model.inject_noise = inject_noise
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        # Weight clipping
        if clip_alpha > 0:
            for m in model.modules():
                if isinstance(m, (nn.Conv2d, nn.Linear)) and m.weight is not None:
                    std = m.weight.data.std()
                    m.weight.data.clamp_(-clip_alpha*std, clip_alpha*std)
        total_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += target.size(0)
    return total_loss/len(loader), 100.*correct/total

def test(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            total_loss += loss.item()
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
            total += target.size(0)
    return total_loss/len(loader), 100.*correct/total

# Data
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010)),
])
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False, num_workers=2)
print(f'Device: {device}')
print(f'Train samples: {len(train_dataset)}')
print(f'Test samples: {len(test_dataset)}')

In [ ]:
# ============================================
# Cell 3: Phase 1 — Baseline FP32 Training (200 epochs)
# ============================================
model_baseline = ResNet32(num_classes=10, inject_noise=False).to(device)
print(f'Parameters: {sum(p.numel() for p in model_baseline.parameters()):,}')

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model_baseline.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)

best_acc = 0
epochs = 200
for epoch in range(epochs):
    start = time.time()
    loss, acc = train_epoch(model_baseline, train_loader, optimizer, criterion, device, inject_noise=False)
    test_loss, test_acc = test(model_baseline, test_loader, criterion, device)
    scheduler.step()
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model_baseline.state_dict(), 'resnet32_baseline_best.pth')
    if epoch % 10 == 0 or epoch == epochs-1:
        print(f'Epoch {epoch:3d}/{epochs} | Train: {acc:.2f}% | Test: {test_acc:.2f}% | Best: {best_acc:.2f}% | Time: {time.time()-start:.1f}s')

print(f'\n✅ Baseline best accuracy: {best_acc:.2f}%')
print(f'IBM paper baseline: 93.87%')

In [ ]:
# ============================================
# Cell 4: Phase 2 — Hardware-Aware Training with Noise Injection
# ============================================
# Initialize from pretrained baseline
model_noise = ResNet32(num_classes=10, inject_noise=True, noise_eta=0.038,
                        protect_first_last=True).to(device)
model_noise.load_state_dict(torch.load('resnet32_baseline_best.pth'))

optimizer_noise = torch.optim.SGD(model_noise.parameters(), lr=0.01,
                                 momentum=0.9, weight_decay=5e-4)
scheduler_noise = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_noise, T_max=50)

best_noise_acc = 0
epochs_noise = 50  # Quick convergence per IBM paper
for epoch in range(epochs_noise):
    start = time.time()
    loss, acc = train_epoch(model_noise, train_loader, optimizer_noise, criterion, device,
                           inject_noise=True, noise_eta=0.038, clip_alpha=2.0)
    test_loss, test_acc = test(model_noise, test_loader, criterion, device)
    scheduler_noise.step()
    if test_acc > best_noise_acc:
        best_noise_acc = test_acc
        torch.save(model_noise.state_dict(), 'resnet32_noise_best.pth')
    if epoch % 5 == 0 or epoch == epochs_noise-1:
        print(f'Noise Epoch {epoch:2d}/{epochs_noise} | Train: {acc:.2f}% | Test: {test_acc:.2f}% | Best: {best_noise_acc:.2f}% | Time: {time.time()-start:.1f}s')

print(f'\n✅ Noise-aware best accuracy: {best_noise_acc:.2f}%')
print(f'IBM paper target: 93.7%')
print(f'Gap to target: {93.7 - best_noise_acc:.2f} percentage points')

In [ ]:
# ============================================
# Cell 5: Phase 3 — Analog Tile Mapping (AIHWKIT)
# ============================================
from aihwkit.nn import AnalogConv2d, AnalogLinear
from aihwkit.simulator.configs import SingleRPUConfig
from aihwkit.simulator.parameters import WeightNoiseType, IOParameters

# PCM noise configuration
rpu_config = SingleRPUConfig()
rpu_config.device.noise_model = WeightNoiseType.PCM_NOISE
rpu_config.device.pcm_noise_params.g_max = 25.0  # μS, per IBM paper

# Convert trained model to analog tiles
def convert_to_analog(module, rpu_config):
    """Recursively convert Conv2d/Linear to analog versions"""
    for name, child in module.named_children():
        if isinstance(child, nn.Conv2d):
            analog = AnalogConv2d(child.in_channels, child.out_channels,
                                 child.kernel_size[0], child.stride[0],
                                 child.padding[0], bias=False,
                                 rpu_config=rpu_config)
            analog.set_weights(child.weight.data)
            setattr(module, name, analog)
        elif isinstance(child, nn.Linear):
            analog = AnalogLinear(child.in_features, child.out_features,
                                 bias=False, rpu_config=rpu_config)
            analog.set_weights(child.weight.data)
            setattr(module, name, analog)
        else:
            convert_to_analog(child, rpu_config)
    return module

# Load best noise-aware model and convert
model_analog = ResNet32(num_classes=10, inject_noise=False).to(device)
model_analog.load_state_dict(torch.load('resnet32_noise_best.pth'))
model_analog = convert_to_analog(model_analog, rpu_config)

# Test on analog hardware
test_loss, test_acc = test(model_analog, test_loader, criterion, device)
print(f'Analog (PCM noise) accuracy: {test_acc:.2f}%')
print(f'Software accuracy: {best_noise_acc:.2f}%')
print(f'Analog degradation: {best_noise_acc - test_acc:.2f}pp')
print(f'IBM paper degradation: <0.2pp')

In [ ]:
# ============================================
# Cell 6: Save results to GitHub (optional)
# ============================================
# Download model weights and results
from google.colab import files
files.download('resnet32_baseline_best.pth')
files.download('resnet32_noise_best.pth')

# Or mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp *.pth /content/drive/MyDrive/laere-research/